# Dimensionality Reduction with PCA

High-dimensional data is expensive to store, slow to train on, and hard to visualize. **Principal Component Analysis (PCA)** compresses many correlated features into a handful of new, uncorrelated features (the *principal components*) while throwing away as little information as possible.

In this notebook we:

1. Load a high-dimensional dataset (`load_digits`, **64 features**) and split it leakage-safely.
2. **Standardize** the features, because PCA is scale-sensitive.
3. Fit PCA and plot the **scree** curve (variance per component) and the **cumulative** explained variance, marking how many components reach **95%**.
4. Visualize the first two principal components as a 2D scatter colored by digit class.
5. Compare a classifier trained on **all 64 original features** vs. one trained on the **PCA-reduced** features: test accuracy, feature-count reduction, and training-time difference.

Everything runs offline with `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `scipy`, `seaborn`, seeded for reproducibility.

In [ ]:
import numpy as np                                       # numeric arrays / linear algebra
import pandas as pd                                      # tidy tables for the comparison summary
import matplotlib.pyplot as plt                          # plotting
import seaborn as sns                                    # nicer default styling for the scatter
import time                                              # wall-clock timing of model training

from sklearn.datasets import load_digits                 # 8x8 handwritten-digit images -> 64 features
from sklearn.model_selection import train_test_split     # train/test split
from sklearn.preprocessing import StandardScaler         # zero-mean / unit-variance scaling
from sklearn.decomposition import PCA                     # the dimensionality reducer itself
from sklearn.pipeline import make_pipeline               # chains scaler -> (PCA) -> classifier, leakage-safe
from sklearn.linear_model import LogisticRegression      # the classifier we compare
from sklearn.metrics import accuracy_score               # test-set accuracy

# One global seed so the split (and anything else stochastic) is reproducible across runs.
SEED = 42
np.random.seed(SEED)

# Cosmetic only: a clean seaborn theme for all figures below.
sns.set_theme(style="whitegrid")

print("Imports OK")

## 1. Load and split a high-dimensional dataset

`load_digits` gives 1797 images of handwritten digits, each an 8x8 grayscale grid **flattened into a 64-dimensional feature vector** (one feature per pixel, values 0-16). The label is the digit 0-9, so this is a 10-class problem.

Adjacent pixels are highly correlated (ink tends to form connected strokes), which is exactly the redundancy PCA exploits.

We hold out 25% for testing and `stratify` on the label so every digit is represented in the same proportion in both splits.

In [ ]:
# X: (1797, 64) pixel-intensity matrix.  y: (1797,) digit label 0-9.
X, y = load_digits(return_X_y=True)
print(f"Full dataset: X={X.shape}, y={y.shape}, classes={np.unique(y)}")

# Split BEFORE any scaling/PCA so the test set stays untouched until final evaluation.
# stratify=y keeps the 0-9 class balance identical in train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 2. PCA intuition, and why we standardize first

**The idea.** PCA finds a new set of axes for the data called *principal components*. They are:

- **orthogonal** (mutually perpendicular, hence uncorrelated), and
- ordered so that the **1st** captures the direction of **maximum variance**, the 2nd captures the most *remaining* variance perpendicular to the 1st, and so on.

Projecting the data onto the first few components keeps most of the "spread" (information) while dropping the rest.

**The eigen/variance connection.** Let $X$ be centered (mean-subtracted) with $m$ samples. Its covariance matrix is

$$\Sigma = \frac{1}{m-1} X^{\top} X .$$

The principal components are the **eigenvectors** of $\Sigma$, and each eigenvalue $\lambda_k$ is the **variance captured** along component $k$. Sorting eigenvectors by descending eigenvalue gives components in order of importance. The fraction of total variance explained by component $k$ is

$$\text{explained variance ratio}_k = \frac{\lambda_k}{\sum_j \lambda_j}.$$

**Why standardize first.** PCA maximizes *variance*, and variance depends on the units/scale of each feature. A feature measured in large numbers would dominate the components purely because its numbers are bigger, not because it is more informative. Standardizing each feature to zero mean and unit variance,

$$x' = \frac{x - \mu}{\sigma},$$

puts every feature on equal footing. Crucially, $\mu$ and $\sigma$ (and the PCA rotation) are learned from the **training set only** — we wrap `StandardScaler` and `PCA` in a `Pipeline` so the test set never leaks into fitting.

In [ ]:
# Pipeline = StandardScaler THEN PCA. When we call .fit(X_train) the scaler learns mu/sigma
# from the TRAIN data, and PCA learns its rotation from those scaled TRAIN features.
# Nothing from the test set is used -> no leakage.
#
# We ask PCA to keep ALL components first (n_components=None => keep 64) so we can inspect the
# full explained-variance spectrum and decide the cutoff ourselves.
pca_pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components=None, random_state=SEED),
)
pca_pipe.fit(X_train)                     # learn scaling + PCA rotation on TRAIN only

pca = pca_pipe.named_steps["pca"]         # grab the fitted PCA step to read its attributes

# explained_variance_ratio_: fraction of total variance carried by each of the 64 components,
# already sorted from most to least important.
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)                   # running total -> cumulative variance explained

# Smallest number of leading components whose cumulative variance first reaches 95%.
# argmax on a boolean array returns the index of the FIRST True; +1 converts index -> count.
n_components_95 = int(np.argmax(cum_evr >= 0.95) + 1)

print(f"Total components available : {len(evr)}")
print(f"Variance in 1st component  : {evr[0]:.3f}")
print(f"Variance in first 2        : {cum_evr[1]:.3f}")
print(f"Components for >=95% var    : {n_components_95}")

## 3. Scree plot and cumulative explained variance

Two complementary views:

- **Scree plot** — variance carried by *each individual* component. It falls off quickly; the "elbow" is where extra components start adding little.
- **Cumulative curve** — the running total. We read off how many components are needed to retain a target fraction (here **95%**) of the total variance.

The vertical/horizontal guides mark the 95% threshold and the component count that first crosses it.

In [ ]:
comp_idx = np.arange(1, len(evr) + 1)          # component numbers 1..64 for the x-axis

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Left: scree plot (per-component explained variance ratio) ---
ax[0].bar(comp_idx, evr, color="#6699cc")
ax[0].set_title("Scree plot - variance per component")
ax[0].set_xlabel("principal component")
ax[0].set_ylabel("explained variance ratio")

# --- Right: cumulative explained variance ---
ax[1].plot(comp_idx, cum_evr, marker=".", color="#333333", label="cumulative variance")
# 95% target line (horizontal) and the component count that first reaches it (vertical).
ax[1].axhline(0.95, color="red", linestyle="--", linewidth=1, label="95% threshold")
ax[1].axvline(n_components_95, color="green", linestyle="--", linewidth=1,
              label=f"{n_components_95} components")
ax[1].set_title("Cumulative explained variance")
ax[1].set_xlabel("number of components")
ax[1].set_ylabel("cumulative explained variance")
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

print(f"{n_components_95} of {len(evr)} components retain >=95% of the variance "
      f"({cum_evr[n_components_95 - 1]:.3f}).")

## 4. Visualizing the first two principal components

We can only look at 2-3 dimensions at once, and the original data lives in 64. Projecting onto the top **two** components gives a 2D map that preserves as much spread as any 2D view can. Coloring each point by its true digit shows whether classes already separate in this compressed space — a quick visual sanity check that PCA kept class-relevant structure.

In [ ]:
# Project the TRAIN data through the fitted scaler+PCA, then keep only the first 2 components.
# transform() applies the SAME train-learned scaling and rotation (no re-fitting).
X_train_2d = pca_pipe.transform(X_train)[:, :2]   # (n_train, 2): [PC1, PC2]

plt.figure(figsize=(7, 6))
# 'tab10' gives 10 distinct colors, one per digit class.
scatter = plt.scatter(
    X_train_2d[:, 0], X_train_2d[:, 1],
    c=y_train, cmap="tab10", s=15, alpha=0.8, edgecolor="k", linewidth=0.2
)
plt.title("Digits projected onto the first 2 principal components")
plt.xlabel(f"PC1 ({evr[0] * 100:.1f}% variance)")
plt.ylabel(f"PC2 ({evr[1] * 100:.1f}% variance)")
# Colorbar acts as the class legend (ticks 0-9).
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.set_label("digit class")
plt.tight_layout()
plt.show()

## 5. Does a model on reduced features perform comparably?

The real test of compression: train the **same** classifier two ways and compare.

- **Baseline** — `StandardScaler -> LogisticRegression` on all **64** features.
- **Reduced** — `StandardScaler -> PCA(95% variance) -> LogisticRegression`.

Both live inside a `Pipeline`, so all preprocessing is fit on train folds only. We report **test accuracy**, the **feature-count reduction**, and the **training-time** difference. A good result: accuracy holds up while the feature count (and often training time) drops.

This is the **accuracy-vs-compactness trade-off** — how much can we shrink the representation before predictive quality starts to suffer?

In [ ]:
# Handy helper: build a pipeline, time its .fit, and score test accuracy.
def train_and_evaluate(pipeline, name):
    t0 = time.perf_counter()               # start the clock
    pipeline.fit(X_train, y_train)          # all preprocessing fit on TRAIN only
    train_time = time.perf_counter() - t0   # wall-clock training seconds
    preds = pipeline.predict(X_test)        # predict on the untouched test set
    acc = accuracy_score(y_test, preds)
    return name, acc, train_time

# max_iter high enough to converge cleanly (avoids a non-convergence warning);
# multinomial softmax logistic regression is the default for multiclass.
clf_kwargs = dict(max_iter=5000, random_state=SEED)

# --- Baseline: scale -> logistic regression on all 64 features ---
full_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(**clf_kwargs),
)

# --- Reduced: scale -> PCA(keep 95% variance) -> logistic regression ---
# Passing a float in (0,1) to n_components tells PCA to keep the smallest number of
# components that reach that cumulative variance -> matches our 95% cutoff automatically.
reduced_pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components=0.95, random_state=SEED),
    LogisticRegression(**clf_kwargs),
)

results = [
    train_and_evaluate(full_pipe, "All 64 features"),
    train_and_evaluate(reduced_pipe, "PCA (95% variance)"),
]

# How many features the reduced model actually kept (read from the fitted PCA step).
n_reduced = reduced_pipe.named_steps["pca"].n_components_

# Assemble a tidy comparison table.
summary = pd.DataFrame(results, columns=["model", "test_accuracy", "train_time_s"])
summary["n_features"] = [X_train.shape[1], n_reduced]
summary = summary[["model", "n_features", "test_accuracy", "train_time_s"]]

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(summary.to_string(index=False))

# Explicit deltas so the trade-off is easy to read.
acc_full, acc_red = summary["test_accuracy"].values
print(f"\nFeatures: {X_train.shape[1]} -> {n_reduced} "
      f"({100 * (1 - n_reduced / X_train.shape[1]):.0f}% fewer)")
print(f"Accuracy change (reduced - full): {acc_red - acc_full:+.4f}")

In [ ]:
# Side-by-side bars make the "comparable accuracy, fewer features" story obvious at a glance.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

colors = ["#6699cc", "#cc6666"]

# Left: test accuracy (zoomed y-axis so tiny differences are visible).
bars = ax[0].bar(summary["model"], summary["test_accuracy"], color=colors)
ax[0].set_ylim(0.9, 1.0)
ax[0].set_title("Test accuracy")
ax[0].set_ylabel("accuracy")
for bar, acc in zip(bars, summary["test_accuracy"]):
    ax[0].text(bar.get_x() + bar.get_width() / 2, acc + 0.002, f"{acc:.3f}", ha="center")

# Right: number of features fed to the classifier.
bars2 = ax[1].bar(summary["model"], summary["n_features"], color=colors)
ax[1].set_title("Feature count into the classifier")
ax[1].set_ylabel("# features")
for bar, n in zip(bars2, summary["n_features"]):
    ax[1].text(bar.get_x() + bar.get_width() / 2, n + 0.5, str(int(n)), ha="center")

plt.tight_layout()
plt.show()

## 6. Takeaways

- **PCA finds orthogonal directions of maximum variance.** They are the eigenvectors of the feature covariance matrix, ordered by eigenvalue (= variance captured).
- **Standardize before PCA.** Otherwise components chase whichever feature happens to have the largest numeric scale rather than the most information. Fit the scaler and PCA on **training data only** (a `Pipeline` enforces this).
- **The scree and cumulative curves** tell you how many components to keep. Here roughly **40 of 64** components retain 95% of the variance.
- **Accuracy-vs-compactness trade-off:** the classifier on the PCA-reduced features scores essentially the same as the one on all 64 features, while feeding far fewer inputs to the model. The dropped components were mostly redundant/noise.

PCA is not free — components are linear mixes of the originals, so they lose direct interpretability, and reduction can occasionally hurt if discarded directions actually carried signal. But when features are correlated (as pixels are), it buys compactness, speed, and denoising at little to no cost in accuracy.